In [28]:
import pandas as pd 
import numpy as np 
import os

In [29]:
synth_img = '/data/7TB/nick/ai_readi_v3/synth_fundus_32/'
df = pd.concat([pd.read_csv(synth_img + x + '.csv', index_col=0) for x in ['train', 'val', 'test']]).reset_index().drop(columns=['index'])

# Create Image Dataset
### retinal_photography

In [30]:
def map_modelname_to_manufacturer_manufacturer_model_name(x):        
    if x['Eidon'] == 1:
        return ['iCare', 'Eidon']
    elif x['Cirrus'] == 1:
        return ['Zeiss', 'Cirrus']
    elif x['Aurora'] == 1:
        return ['Optomed', 'Aurora']
    elif x['Spectralis'] == 1:
        return ['Heidelberg', 'Spectralis']
    elif x['Triton'] == 1:
        return ['Topcon', 'Triton']
    elif x['Maestro2'] == 1:
        return ['Topcon', 'Maestro2']
    
def map_laterality(x):        
    if x['R'] == 1:
        return 'R'
    else: 
        return 'L'
    
def map_anatomic_region(x):
    for region in ['Macula', 'Macula or Optic Disc', 'Mosaic', 'Nasal', 'Optic Disc', 'Temporal Periphery', 'Wide Field', 'Infrared Reflectance']:
        if x[region] == 1:
            return region

def map_image_type(x):
    for img_type in ['Autofluorescence', 'Color Photography', 'Infrared Reflectance']:
        if x[img_type] == 1:
            return img_type

def map_color_dim(x):
    if x['Infrared Reflectance'] == 1:
        return 0
    else:
        return 3

In [31]:
base = '/data/7TB/nick/synth_ai_readi_fundus'

# Top level directory
os.makedirs(base, exist_ok=True)

# Modality-specific directories
os.makedirs(os.path.join(base, 'clinical_data'), exist_ok=True)
os.makedirs(os.path.join(base, 'retinal_photography'), exist_ok=True)

# Image-type directories
os.makedirs(os.path.join(base, 'retinal_photography', 'cfp'), exist_ok=True)
os.makedirs(os.path.join(base, 'retinal_photography', 'faf'), exist_ok=True)
os.makedirs(os.path.join(base, 'retinal_photography', 'ir'), exist_ok=True)


In [ ]:
manifest = df.copy()
df['participant_id'] = np.nan
df.loc[:, ['manufacturer', 'manufacturers_model_name']] = np.concatenate(df.apply(map_modelname_to_manufacturer_manufacturer_model_name, axis=1), axis=0).reshape(-1,2)
df.loc[:, ['laterality']] = df.apply(map_laterality, axis=1)
df.loc[:, ['anatomic_region']] = df.apply(map_anatomic_region, axis=1)
df.loc[:, ['imaging']] = df.apply(map_image_type, axis=1)
df.loc[:, 'height'] = 256
df.loc[:, 'width'] = 256
df.loc[:, 'color_channel_dimension'] = df.apply(map_color_dim, axis=1)

np.int64(0)

In [34]:
import pydicom
from pydicom.dataset import Dataset, FileDataset
from pydicom.uid import generate_uid, ExplicitVRLittleEndian, OphthalmicPhotography8BitImageStorage
from PIL import Image
import datetime

def create_ophthalmic_dicom(data, patient_id):
    patient_name="DOE^JOHN"

    # Select directory based on image type
    if data['Color Photography'] == 1:
        mod = 'cfp'
    elif data['Infrared Reflectance'] == 1:
        mod = 'ir'
    else:
        mod = 'faf'

    # select directory based on equipment
    manu = (data['manufacturer'] + '_' + data['manufacturers_model_name']).lower()

    #
    output_dicom_path = os.path.join(base, 'retinal_photography', mod, manu, str(patient_id))
    os.makedirs(output_dicom_path, exist_ok=True)

    # Read Image
    img = Image.open(data['filepath'])
    pixel_array = np.array(img)

    rows, cols, samples_per_pixel = pixel_array.shape

    # Create File Meta Information
    file_meta = pydicom.Dataset()
    file_meta.MediaStorageSOPClassUID = OphthalmicPhotography8BitImageStorage
    file_meta.MediaStorageSOPInstanceUID = generate_uid()
    file_meta.ImplementationClassUID = generate_uid()

    # Create DICOM Dataset
    ds = FileDataset(output_dicom_path, {}, file_meta=file_meta, preamble=b"\0" * 128)

    # Set the transfer syntax
    ds.is_little_endian = True
    ds.is_implicit_VR = False

    # Patient and Study Information
    ds.PatientName = patient_name
    ds.PatientID = str(patient_id)
    ds.StudyInstanceUID = generate_uid()
    ds.SeriesInstanceUID = generate_uid()
    ds.SOPInstanceUID = file_meta.MediaStorageSOPInstanceUID
    ds.SOPClassUID = file_meta.MediaStorageSOPClassUID

    # Date and Time
    dt = datetime.datetime.now()
    ds.StudyDate = dt.strftime('%Y%m%d')
    ds.StudyTime = dt.strftime('%H%M%S')

    # Modality and Image Type
    ds.Manufacturer = data['manufacturer']
    ds.ManufacturerModelName = data['manufacturers_model_name']
    ds.ImageType = ['ORIGINAL', 'PRIMARY']
    ds.PhotometricInterpretation = "RGB" if data['imaging'] != 'Infrared Reflectance' else 'MONOCHROME1'
    ds.SamplesPerPixel = samples_per_pixel
    ds.Rows = rows
    ds.Columns = cols
    
    ds.BitsAllocated = 8
    ds.BitsStored = 8
    ds.HighBit = 7
    ds.PixelRepresentation = 0
    ds.PlanarConfiguration = 0 

    # Ophthalmic-specific tags (simplified example)
    ds.ImageLaterality = data['laterality']  # 'R'ight, 'L'eft, 'U'nspecified
    ds.NumberOfFrames = 1

    # Set the Pixel Data
    ds.PixelData = pixel_array.tobytes()

    # Save the DICOM file
    ds.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

    filename = "_".join([str(patient_id), data['manufacturers_model_name'], data['anatomic_region'], data['imaging'], ds.SOPInstanceUID]).lower().replace(' ', '_') + '.dcm'
    output_dicom_path = os.path.join(output_dicom_path, filename)
    
    ds.save_as(output_dicom_path)
    #print(f"DICOM saved to: {output_dicom_path}")

    return output_dicom_path.split(base+'/')[1], ds.SOPInstanceUID

In [35]:
from tqdm import tqdm

df.loc[:, 'sop_instance_uid'] = ''
for i, row in tqdm(df.iterrows(), total=len(df)):
    fp = create_ophthalmic_dicom(row, i)
    df.loc[i, 'filepath'] = fp[0]
    df.loc[i, 'sop_instance_uid'] = fp[1]

100%|██████████| 93856/93856 [04:35<00:00, 341.14it/s]


In [36]:
df.loc[:,'participant_id'] = df.index.values
df['participant_id'] = df['participant_id'].astype(int)
df[['participant_id','manufacturer','manufacturers_model_name','laterality','anatomic_region','imaging','height','width','color_channel_dimension', 'sop_instance_uid', 'filepath']].to_csv(os.path.join(base, 'retinal_photography') + '/manifest.tsv', sep='\t')
df[['participant_id','manufacturer','manufacturers_model_name','laterality','anatomic_region','imaging','height','width','color_channel_dimension', 'sop_instance_uid', 'filepath']]

,participant_id,manufacturer,manufacturers_model_name,laterality,anatomic_region,imaging,height,width,color_channel_dimension,sop_instance_uid,filepath
0,0,Topcon,Maestro2,L,Macula,Infrared Reflectance,256,256,0,1.2.826.0.1.3680043.8.498.15878151434457714965...,retinal_photography/ir/topcon_maestro2/0/0_mae...
1,1,Zeiss,Cirrus,R,Optic Disc,Infrared Reflectance,256,256,0,1.2.826.0.1.3680043.8.498.66201851658259575010...,retinal_photography/ir/zeiss_cirrus/1/1_cirrus...
2,2,Zeiss,Cirrus,R,Optic Disc,Infrared Reflectance,256,256,0,1.2.826.0.1.3680043.8.498.96807217527126325658...,retinal_photography/ir/zeiss_cirrus/2/2_cirrus...
3,3,iCare,Eidon,L,Temporal Periphery,Color Photography,256,256,3,1.2.826.0.1.3680043.8.498.54011938252375144331...,retinal_photography/cfp/icare_eidon/3/3_eidon_...
4,4,Heidelberg,Spectralis,L,Infrared Reflectance,Infrared Reflectance,256,256,0,1.2.826.0.1.3680043.8.498.70110481962480440943...,retinal_photography/ir/heidelberg_spectralis/4...
...,...,...,...,...,...,...,...,...,...,...,...
93851,93851,Topcon,Triton,L,Optic Disc,Color Photography,256,256,3,1.2.826.0.1.3680043.8.498.95905715694459222032...,retinal_photography/cfp/topcon_triton/93851/93...
93852,93852,iCare,Eidon,R,Macula,Color Photography,256,256,3,1.2.826.0.1.3680043.8.498.89919491847636760828...,retinal_photography/cfp/icare_eidon/93852/9385...
93853,93853,Heidelberg,Spectralis,R,Optic Disc,Infrared Reflectance,256,256,0,1.2.826.0.1.3680043.8.498.45995440623647107754...,retinal_photography/ir/heidelberg_spectralis/9...
93854,93854,Heidelberg,Spectralis,R,Optic Disc,Infrared Reflectance,256,256,0,1.2.826.0.1.3680043.8.498.84041864209201965934...,retinal_photography/ir/heidelberg_spectralis/9...


# Clinical data (OMOP) for synthetic image dataset
### (only contains observation.csv)

In [37]:
obs = pd.read_csv('/data/7TB/nick/ai_readi/clinical_data/observation.csv')

In [38]:
codes = {'AMD': 374028, 'DR':4174977, 'GL': 437541}
counter = 0
new_obs = []
for name, code in codes.items():
    temp = obs[obs['qualifier_concept_id'] == code]
    cols_same = {col:[] for col in temp.columns if len(temp[col].value_counts()) == 1}
    cols_new = {col:[] for col in temp.columns if len(temp[col].value_counts()) != 1}

    for i, row in df.iterrows():
        for col in cols_same:
            cols_same[col].append(temp.iloc[0][col])
        
        cols_new['observation_id'].append(int(counter))
        cols_new['person_id'].append(int(row['participant_id']))

        now = datetime.datetime.now()
        omop_date = now.date()                     # date: YYYY-MM-DD
        omop_datetime = now.strftime('%Y-%m-%d %H:%M:%S')  # datetime: string format
    
        cols_new['observation_date'].append(omop_date)
        cols_new['observation_datetime'].append(omop_datetime)
        cols_new['value_as_number'].append(int(row[name]))
        cols_new['value_as_string'].append(str(row[name]))
        cols_new['value_as_concept_id'].append('4188539' if row[name] == 1 else '45878245')
        cols_new['visit_occurrence_id'].append(int(row['participant_id']))

        counter += 1

    merged = {**cols_new, **cols_same}
    new_obs.append(pd.DataFrame(merged)[obs.columns])

new_obs = pd.concat(new_obs)

In [39]:
new_obs.to_csv(os.path.join(base, 'clinical_data') + '/observation.csv')
